In [ ]:
import sys
from pathlib import Path


p = Path.cwd()
while p != p.parent and not (p / "kxor_code").exists():
    p = p.parent

sys.path.insert(0, str(p))
print("Added to sys.path:", p)

Added to sys.path: /Users/daphnejanissen/Documents/DSDM/Quantum Research Project/QuarticSpeedupK-XOR


In [11]:
from pathlib import Path
Path("data/problem_instances").mkdir(parents=True, exist_ok=True)

In [12]:
import pickle

with open("theorem_results.pkl", "rb") as f:
    results = pickle.load(f)


In [13]:
import os
from kxor_code.problem_set_generation.kxor_dataset_generator import KXORDatasetGenerator

# Filter theorem-feasible sets + extra constraint m < 255
filtered = [
    r for r in results
    if int(r["n"]) < 31
    and float(r["failure_probability_bound"]) < 0.1
    and int(r["m"]) < 255
]

print(f"Using {len(filtered)} theorem-feasible sets (n < 31, m < 255, fail_bound < 0.1)")

# Convert them to explicit (n, k, m, rho) parameter tuples for your generator and set rho grid
rho_step = 0.2
rho_values = [round(i * rho_step, 10) for i in range(1, int(1.0 / rho_step) + 1)]  # 0.2..1.0

# Random instances (rho=0.0)
include_random = True

params = []
for r in filtered:
    n, k, m = int(r["n"]), int(r["k"]), int(r["m"])

    if include_random:
        params.append((n, k, m, 0.0))  # random instance

    for rho in rho_values:
        params.append((n, k, m, float(rho)))  # planted instances

# Remove duplicates
params = sorted(set(params))

print(f"Total instances to generate (unique n,k,m,rho combos): {len(params)}")

# Run your existing pipeline
folder_path = "data/problem_instances"
os.makedirs(folder_path, exist_ok=True)

generator = KXORDatasetGenerator()
generator.generate_kxor_dataset_explicit_params(
    folder_path=folder_path,
    params=params,
    seed=42,
    zip=False
)

print("Done. Dataset written to:", folder_path)


Using 367 theorem-feasible sets (n < 31, m < 255, fail_bound < 0.1)
Total instances to generate (unique n,k,m,rho combos): 846
Done. Dataset written to: data/problem_instances
